In [ ]:
import pandas as pd
import datetime
import tushare as ts
import os
from xtquant import xtdatacenter as xtdc
from xtquant import xtdata
import random
import pandas as pd
import numpy as np
import yaml
def load_config():
    with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)
    return config
config = load_config()
TUSHARE_API_KEY = config["api"]["tushare_api_key"]
XTQUANT_API_KEY = config["api"]["xtquant_api_key"]
ts.set_token(TUSHARE_API_KEY)
pro = ts.pro_api(TUSHARE_API_KEY)
# xtdc.set_token("857b1fa0068599ccb191546f86bac28e09d8a727") # 用户试用已过期
xtdc.set_token(XTQUANT_API_KEY)
xtdc.set_data_home_dir("../xtquant_tushare/srv/xuntoudata/data")
xtdc.init(False)
actual_port = random.randint(58600,58700)
xtdc.listen(port=actual_port)
print(f"服务启动,开放端口：{actual_port}")

服务启动,开放端口：58675


In [15]:
# 1.获取从基准日到最近交易日的日期列表
weight_start_date = '20260227'
# index_code = '000852.SH'
# index_code = '000901.SH'
# index_code = '000905.SH'
# index_code = '000903.SH'
index_code = '000300.SH'
file_dir = ""
for filedir in os.listdir(f'./ref/project_/cal_index_data/'):
    if index_code in filedir:
        # print(filedir)
        file_dir = filedir
        break
xtdata.download_history_data(index_code,'1d',weight_start_date, "")


{'000300.SH': {'start_time': datetime.datetime(2014, 5, 29, 0, 0),
  'end_time': datetime.datetime(2026, 3, 25, 0, 0),
  'count': 2874}}

In [ ]:
def get_prev_trade_date(date):
    prev_day =(pd.to_datetime(date) - pd.Timedelta(days=1)).strftime('%Y%m%d')
    while True:
        if int(pro.trade_cal(exchange='SSE', start_date=prev_day, end_date=prev_day)['is_open'][0]) == 1:
            return prev_day
        else :
            prev_day =(pd.to_datetime(prev_day) - pd.Timedelta(days=1)).strftime('%Y%m%d')
now_date = pd.Timestamp(datetime.datetime.now()).date().strftime('%Y%m%d')
last_trade_day = get_prev_trade_date(now_date)
# dividend_info_df = get_dividend_info_(index_code,weight_start_date)
# dividend_stock_code_dict = get_dividend_stock_dict(dividend_info_df)
# newest_price_df = get_stocks_closePrice_info(index_code,weight_start_date,last_trade_day)
# cal_weight_df = cal_adjust_weight_and_write_to_csv(index_code,weight_start_date,last_trade_day,dividend_info_df,dividend_stock_code_dict,newest_price_df)


In [4]:
# 2.找出交易日期列表中存在的除权除息日并且合并原有除权除息数据？
# 这个地方不太懂，那个dividend文件里的各项的具体含义不太理解
import os
def get_dividend_info_(index_code,weight_start_date):
    # 获取成分股从给定日期weight_start_date到当前日期的所有分红送股数据
    data = xtdata.get_market_data_ex([],[index_code],period = '1d', start_time = weight_start_date, end_time = '', count = 0) # count 设置为1，使返回值只包含最新行情
    trade_date_list = list(data[index_code].index)
    
    dividend_info_file = f"./ref/project_/cal_index_data/{file_dir}/dividend_data_{index_code}_{weight_start_date}.csv"
    # dividend_info_file = f"./ref/project_/cal_index_data/dividend_data_{index_code}_{weight_start_date}_副本.csv"
    if os.path.exists(dividend_info_file) and dividend_info_file != "":
        dividend_info_df = pd.read_csv(dividend_info_file, dtype={'ex_date': str})
        existed_date_set = set(dividend_info_df['ex_date'].tolist())
    else:
        dividend_info_df = pd.DataFrame()
        existed_date_set = set()
    for date in trade_date_list:
        if date == weight_start_date or date in existed_date_set:
            continue
        dividend_df = pro.dividend(ex_date=date)
        if not dividend_df.empty:
            dividend_df_clean = dividend_df.dropna(axis=1, how='all')
            dividend_info_df = pd.concat([dividend_info_df, dividend_df_clean], ignore_index=True)
    dividend_info_df.to_csv(dividend_info_file, index=False)
    return dividend_info_df
dividend_info_df = get_dividend_info_(index_code,weight_start_date)
dividend_info_df

,ts_code,end_date,ann_date,div_proc,stk_div,cash_div,cash_div_tax,record_date,ex_date,pay_date,imp_ann_date,stk_co_rate,div_listdate
0,002107.SZ,20251231,20260120,实施,0.0,0.146,0.146,20260304,20260305,20260305.0,20260227,NaN,NaN
1,603048.SH,20250630,20250830,实施,0.0,0.070,0.070,20260304,20260305,20260305.0,20260227,NaN,NaN
2,300803.SZ,20251231,20260131,实施,0.0,0.080,0.080,20260305,20260306,20260306.0,20260228,NaN,NaN
3,000905.SZ,20250930,20260224,实施,0.0,0.030,0.030,20260305,20260306,20260306.0,20260228,NaN,NaN
4,300957.SZ,20250930,20260211,实施,0.0,0.300,0.300,20260309,20260310,20260310.0,20260304,NaN,NaN
5,000788.SZ,20250930,20260105,实施,0.0,0.168,0.168,20260309,20260310,20260310.0,20260303,NaN,NaN
6,002128.SZ,20250930,20260122,实施,0.0,0.300,0.300,20260309,20260310,20260310.0,20260228,NaN,NaN
7,000908.SZ,20260113,20260113,实施,1.0,0.000,0.000,20260310,20260311,NaN,20260304,1.0,20260311.0
8,300651.SZ,20250930,20251226,实施,0.0,0.100,0.100,20260310,20260311,20260311.0,20260304,NaN,NaN
9,920207.BJ,20250930,20260212,实施,0.0,0.150,0.150,20260313,20260316,20260316.0,20260310,NaN,NaN


In [5]:
# 统计每个股票送股的次数
def get_dividend_stock_dict(dividend_info_df):
    dividend_stock_code_list = dividend_info_df[dividend_info_df['stk_div'] != 0].reset_index(drop=True)['ts_code'].tolist()
    dividend_stock_code_dict = {}
    for ts_code in dividend_stock_code_list:
        if ts_code not in dividend_stock_code_dict:
            dividend_stock_code_dict[ts_code] = 1
        else:
            dividend_stock_code_dict[ts_code] += 1
    return dividend_stock_code_dict
dividend_stock_code_dict = get_dividend_stock_dict(dividend_info_df)
dividend_stock_code_dict

{'000908.SZ': 1}

In [6]:
# 3.获取成分股在上一个交易日的收盘价，如果有股票在之前就停牌，就用最近一个交易日的收盘价补齐
def get_stocks_closePrice_info(index_code,weight_start_date,last_trade_day):
    index_df = pd.read_csv(f'./ref/project_/cal_index_data/{file_dir}/{index_code}_{weight_start_date} - 副本.csv')
    code_list = index_df['code'].tolist()
    len(code_list)
    code_list_ = ','.join(code_list)
    newest_price_df = ts.pro_bar(ts_code=code_list_, start_date=last_trade_day, end_date=last_trade_day).iloc[::-1].reset_index(drop=True)
    
    # 获取出现特殊事件的股票
    code_set = set(code_list)
    returned_set = set(newest_price_df['ts_code'].tolist())
    missing_codes = code_set - returned_set

    # 将last_trade_day中缺失的股票的最近一个交易日的数据补齐
    for code in missing_codes:
        print(code)
        pre_trade_day = get_prev_trade_date(last_trade_day)
        missing_code_df = ts.pro_bar(ts_code=code, start_date=pre_trade_day, end_date=pre_trade_day).iloc[::-1].reset_index(drop=True)
        while len(missing_code_df) == 0:
            pre_trade_day = get_prev_trade_date(pre_trade_day)
            missing_code_df = ts.pro_bar(ts_code=code, start_date=pre_trade_day, end_date=pre_trade_day).iloc[::-1].reset_index(drop=True)
        newest_price_df = pd.concat([newest_price_df, missing_code_df], axis=0)
    return newest_price_df
newest_price_df = get_stocks_closePrice_info(index_code,weight_start_date,last_trade_day)
newest_price_df


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,688800.SH,20260324,86.99,88.00,83.30,87.10,NaN,NaN,NaN,91666.71,785419.550
1,688798.SH,20260324,67.20,67.49,65.52,67.36,87.10,-19.74,-22.66,15939.62,105872.957
2,688789.SH,20260324,68.57,70.60,68.57,70.17,67.36,2.81,4.17,10553.92,73540.961
3,688779.SH,20260324,10.66,10.82,10.12,10.62,70.17,-59.55,-84.87,372633.97,390346.420
4,688776.SH,20260324,85.36,87.85,83.60,87.58,10.62,76.96,724.67,23602.71,201756.445
...,...,...,...,...,...,...,...,...,...,...,...
995,000030.SZ,20260324,4.76,4.86,4.70,4.85,7.46,-2.61,-34.99,128444.80,61439.622
996,000029.SZ,20260324,18.89,19.30,18.51,19.29,4.85,14.44,297.73,69183.40,131541.000
997,000028.SZ,20260324,24.65,25.23,24.49,25.18,19.29,5.89,30.53,52521.48,130642.195
998,000019.SZ,20260324,7.02,7.25,6.84,7.25,25.18,-17.93,-71.21,182358.56,128693.960


In [7]:
# 4.计算最新权重
def cal_adjust_weight_and_write_to_csv(index_code,weight_start_date,last_trade_day,dividend_info_df,dividend_stock_code_dict,newest_price_df):
    # 添加除权价格
    index_df = pd.read_csv(f'./ref/project_/cal_index_data/{file_dir}/{index_code}_{weight_start_date} - 副本.csv')
    code_list = index_df['code'].tolist()
    cal_weight_df = index_df.copy(deep=True)
    price_map = newest_price_df.set_index('ts_code')['close']
    cal_weight_df[f'{last_trade_day}close'] = cal_weight_df['code'].map(price_map)

    for dividend_stock_code in dividend_stock_code_dict:
        if dividend_stock_code not in code_list:
            continue
        cq_price = cal_weight_df.loc[dividend_stock_code, f'{last_trade_day}close']
        stock_dividend_rate = dividend_info_df.loc[
            dividend_info_df['ts_code'] == dividend_stock_code, 'stk_div'
        ].values[0]
        adjust_cq_price = (1 + stock_dividend_rate) * cq_price
        cal_weight_df.loc[dividend_stock_code, f'{last_trade_day}close'] = adjust_cq_price
    # 计算
    # (当天调整除权收盘价-公告日除权收盘价)/公告日除权收盘价=调整除权涨跌幅
    cal_weight_df[f'{last_trade_day}chg'] = ((cal_weight_df[f'{last_trade_day}close'] - cal_weight_df[f'{weight_start_date}close']) / cal_weight_df[f'{weight_start_date}close'])  
    # (1+调整除权涨跌幅)*公告权重=当天权重
    cal_weight_df[f'{last_trade_day}weight'] = ((1 + cal_weight_df[f'{last_trade_day}chg']) * cal_weight_df[f'{weight_start_date}weight'])  
    # 权重总和
    total_weight = cal_weight_df[f'{last_trade_day}weight'].sum()  
    scale_factor = 100 / total_weight  # 将权重重新归于100
    cal_weight_df[f'{last_trade_day}weight'] = cal_weight_df[f'{last_trade_day}weight'] * scale_factor

    # data_dir = './ref/project_/cal_index_data'
    cal_weight_df.to_csv(os.path.join(f'./ref/project_/cal_index_data/{file_dir}', f'cal_weight_df_{index_code}_{last_trade_day}_副本.csv'),index=False)
    
    return cal_weight_df
cal_weight_df = cal_adjust_weight_and_write_to_csv(index_code,weight_start_date,last_trade_day,dividend_info_df,dividend_stock_code_dict,newest_price_df)
cal_weight_df

,code,20260227weight,20260227close,20260324close,20260324chg,20260324weight
0,000012.SZ,0.073,5.03,4.45,-0.115308,0.072746
1,000019.SZ,0.025,7.20,7.25,0.006944,0.028355
2,000028.SZ,0.053,25.83,25.18,-0.025165,0.058197
3,000029.SZ,0.066,23.21,19.29,-0.168893,0.061786
4,000030.SZ,0.030,5.56,4.85,-0.127698,0.029477
...,...,...,...,...,...,...
995,688776.SH,0.065,113.94,87.58,-0.231350,0.056277
996,688779.SH,0.142,11.55,10.62,-0.080519,0.147069
997,688789.SH,0.099,86.59,70.17,-0.189629,0.090367
998,688798.SH,0.079,80.44,67.36,-0.162606,0.074516


### 以下是草稿，用来验证和给的参考代码跑出来的文件是否一致。我自己跑出来的csv都叫副本

In [8]:
W_1 = pd.read_csv(f'ref/project_/cal_index_data/{file_dir}/cal_weight_df_{index_code}_{last_trade_day}_副本.csv')
W_2 = pd.read_csv(f'ref/project_/cal_index_data/{file_dir}/cal_weight_df_{index_code}_{last_trade_day}.csv')
import numpy as np

# 设置绝对容差和相对容差
atol = 1e-11  # 绝对容差（例如 0.000001）
rtol = 1e-10  # 相对容差

if np.allclose(W_1[f'{last_trade_day}weight'], W_2[f'{last_trade_day}weight'], rtol=rtol, atol=atol):
    print("两列数据在容差范围内相等")
else:
    print("两列数据不完全相等")
if np.allclose(W_1[f'{last_trade_day}close'], W_2[f'{last_trade_day}close'], rtol=rtol, atol=atol):
    print("两列数据在容差范围内相等")
else:
    print("两列数据不完全相等")

FileNotFoundError: [Errno 2] No such file or directory: 'ref/project_/cal_index_data/中证1000_000852.SH/cal_weight_df_000852.SH_20260324.csv'